# **Preprocessing | MASSIVE-Agents**

---

**Project:** KP Agentic AI + LLM  
**Dataset:** `massive-full-converted-all-langs-with-id` · `en-US` & `id-ID`  

**Notes:** Data asli di `data/raw` tidak diubah. Hasil pembagian dataset masih bersifat awal dan perlu validasi sebelum evaluasi model.

In [1]:
from __future__ import annotations

import csv
import hashlib
import itertools
import json
import platform
import random
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Platform:", platform.system())
print("Working directory:", Path.cwd())

Python: 3.13.12
Pandas: 3.0.6
Platform: Windows
Working directory: d:\KP-Agentic-AI\notebooks


## **1 · Pengaturan Dataset dan Pembagian Awal**

---

Dataset menggunakan Bahasa Inggris (`en-US`) dan Indonesia (`id-ID`). Pembagian awal ditetapkan sebesar 30% grup untuk *development* dan 70% untuk *test*, menggunakan `RANDOM_SEED` agar hasilnya konsisten.

Seluruh katalog fungsi asli tetap dipertahankan. Data dengan masalah skema atau ground truth ditandai untuk review, bukan langsung dihapus.

In [2]:
DATASET_FOLDER = "massive-full-converted-all-langs-with-id"
LANGUAGES = ["en-US", "id-ID"]
PROJECT_ROOT_OVERRIDE = None  # Misalnya Path(r"D:\KP-Agentic-AI") jika pencarian otomatis gagal
RANDOM_SEED = 20261020
DEV_GROUP_FRACTION = 0.30
STRICT_MANIFEST_IF_PRESENT = True

assert 0 < DEV_GROUP_FRACTION < 1

def locate_project(override=None):
    candidates = [Path(override).expanduser()] if override else [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / "data" / "raw" / DATASET_FOLDER).is_dir():
            return p.resolve()
    raise FileNotFoundError("Folder data/raw/massive-full-converted-all-langs-with-id tidak ditemukan. Buka folder KP-Agentic-AI di VS Code.")

ROOT = locate_project(PROJECT_ROOT_OVERRIDE)
RAW = ROOT / "data" / "raw" / DATASET_FOLDER
OUT = ROOT / "results" / "preprocessing"
PROCESSED = ROOT / "data" / "processed" / "massive_agents"
OUT.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

for lang in LANGUAGES:
    for filename in ("input.jsonl", "golden.jsonl"):
        path = RAW / lang / filename
        if not path.is_file():
            raise FileNotFoundError(f"File wajib tidak ditemukan: {path}")

print("Project:", ROOT)
print("Raw:", RAW)
print("Hasil audit:", OUT)
print("Dataset olahan:", PROCESSED)
print("Random seed:", RANDOM_SEED, "| dev fraction per grup:", DEV_GROUP_FRACTION)

Project: D:\KP-Agentic-AI
Raw: D:\KP-Agentic-AI\data\raw\massive-full-converted-all-langs-with-id
Hasil audit: D:\KP-Agentic-AI\results\preprocessing
Dataset olahan: D:\KP-Agentic-AI\data\processed\massive_agents
Random seed: 20261020 | dev fraction per grup: 0.3


## **2 · Pemeriksaan Sumber Data dan Checksum**

---

Checksum SHA-256 dibandingkan dengan hasil Data Inspection untuk memastikan file sumber tidak berubah. Jika checksum berbeda, proses dihentikan agar dataset yang digunakan tetap konsisten.

In [3]:
def sha256_file(path: Path, buffer_size=1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(buffer_size), b""):
            h.update(chunk)
    return h.hexdigest()

raw_manifest = pd.DataFrame([
    {"language_code": lang, "filename": fn,
     "bytes": (RAW/lang/fn).stat().st_size,
     "sha256": sha256_file(RAW/lang/fn)}
    for lang in LANGUAGES for fn in ("input.jsonl", "golden.jsonl")
])
prior_manifest_path = ROOT / "results" / "data_inspection" / "file_manifest.csv"
if prior_manifest_path.exists():
    earlier = pd.read_csv(prior_manifest_path, dtype=str)
    if {"language_code", "filename", "sha256"}.issubset(earlier.columns):
        compare = raw_manifest.merge(
            earlier[["language_code", "filename", "sha256"]],
            on=["language_code", "filename"], how="left",
            suffixes=("_now", "_inspection"), validate="one_to_one"
        )
        compare["same_as_inspection"] = compare.sha256_now.eq(compare.sha256_inspection)
        display(compare[["language_code", "filename", "same_as_inspection"]])
        if STRICT_MANIFEST_IF_PRESENT and not compare.same_as_inspection.all():
            raise RuntimeError("Sumber berubah sejak Data Inspection atau manifest belum cocok. Periksa sebelum lanjut.")
    else:
        print("Manifest sebelumnya tidak memuat kolom hash standar; verifikasi manual diperlukan.")
else:
    print("Manifest Data Inspection tidak ditemukan; checksum tahap ini tetap dicatat.")

display(raw_manifest)

,language_code,filename,same_as_inspection
0,en-US,input.jsonl,True
1,en-US,golden.jsonl,True
2,id-ID,input.jsonl,True
3,id-ID,golden.jsonl,True


,language_code,filename,bytes,sha256
0,en-US,input.jsonl,81866563,2a43ab25bec162ae5c77e340d7afe0cdafa3dcd8972ca3...
1,en-US,golden.jsonl,256566,aa3d369cde3d1d92c6886644d49ac797f6cfaab46d3509...
2,id-ID,input.jsonl,55993920,588a109dddb83d2da9caadfe95a9415e33728353a3cb26...
3,id-ID,golden.jsonl,175693,f7717e3edc3bda9b7843e62cdbc23d5f3b055542e6f2a1...


## **3 · Membaca dan Memeriksa Pasangan JSONL**

---

Data dalam `input.jsonl` dicocokkan dengan `golden.jsonl` berdasarkan urutan baris. Jika ditemukan masalah pembacaan atau ground truth yang tidak cocok, proses dihentikan tanpa mengubah data asli.

Field `language` pada data sumber berisi bahasa pemrograman, bukan kode bahasa Inggris atau Indonesia.

In [4]:
def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def text_key(question: str):
    """Untuk grouping duplikasi saja; teks asli tidak berubah."""
    return " ".join(question.split()).casefold()

records = []
parsing_errors = []
for lang in LANGUAGES:
    a = RAW/lang/"input.jsonl"
    b = RAW/lang/"golden.jsonl"
    with a.open("r", encoding="utf-8-sig") as fi, b.open("r", encoding="utf-8-sig") as fg:
        for line_no, (input_line, golden_line) in enumerate(itertools.zip_longest(fi, fg), 1):
            row_key = f"{lang}:{line_no:07d}"
            if input_line is None or golden_line is None:
                parsing_errors.append((row_key, "input_and_golden_line_count_mismatch"))
                continue
            try:
                row, golden = json.loads(input_line), json.loads(golden_line)
            except (ValueError, TypeError) as exc:
                parsing_errors.append((row_key, "json_parse_error:" + str(exc)))
                continue
            if not isinstance(row, dict) or not isinstance(golden, dict):
                parsing_errors.append((row_key, "input_or_golden_not_object"))
                continue
            if row.get("gt") != golden:
                parsing_errors.append((row_key, "gt_differs_from_paired_golden"))
                continue
            q = row.get("question")
            if not isinstance(q, str) or not q.strip():
                parsing_errors.append((row_key, "missing_or_invalid_question"))
                continue
            if row.get("id") is None or not isinstance(row.get("gt"), dict):
                parsing_errors.append((row_key, "missing_id_or_invalid_gt"))
                continue
            if not isinstance(row.get("function"), list):
                parsing_errors.append((row_key, "function_catalog_not_list"))
                continue
            gt = row["gt"]
            records.append({
                "row_key": row_key, "language_code": lang, "line_number": line_no,
                "record_id": str(row["id"]), "question": q,
                "question_cmp": text_key(q), "gt": gt,
                "gt_signature": canonical_json(gt),
                "gt_names": tuple(sorted(gt.keys())),
                "test_category": row.get("test_category"),
                "source_language_field": row.get("language"),
            })

if parsing_errors:
    pd.DataFrame(parsing_errors, columns=["row_key", "problem"]).to_csv(
        OUT/"fatal_parsing_errors.csv", index=False, encoding="utf-8-sig")
    raise RuntimeError(f"Ditemukan {len(parsing_errors)} masalah struktur/pasangan input–golden. Periksa fatal_parsing_errors.csv; ekspor dihentikan.")

if not records:
    raise RuntimeError("Dataset terbaca kosong.")

meta = pd.DataFrame([{k:v for k,v in rec.items() if k not in ("gt", "gt_names")}
                     | {"target_function": ";".join(rec["gt_names"])} for rec in records])
assert meta.row_key.is_unique
print("Total sampel terverifikasi secara struktur:", len(records))
display(meta.groupby("language_code").size().rename("records").reset_index())
display(meta[["row_key", "record_id", "question", "target_function", "test_category"]].head(6))

Total sampel terverifikasi secara struktur: 3287


,language_code,records
0,en-US,1952
1,id-ID,1335


,row_key,record_id,question,target_function,test_category
0,en-US:0000001,0,wake me up at five am this week,alarm.set,multiple_function
1,en-US:0000002,3,quiet,audio.volume_mute,multiple_function
2,en-US:0000003,8,pink is all we need,iot.hue_lightchange,multiple_function
3,en-US:0000004,14,and the darkness has fallen,iot.hue_lighton,multiple_function
4,en-US:0000005,19,olly turn the lights off in the bedroom,iot.hue_lightoff,multiple_function
5,en-US:0000006,47,what's the time in australia,datetime.query,multiple_function


## **4 · Pemeriksaan Ground Truth dan Skema Fungsi**

---

Nama fungsi dan argumen dalam `gt` diperiksa berdasarkan katalog fungsi yang tersedia pada setiap data, termasuk parameter wajib.

Argumen yang tidak sesuai skema ditandai untuk review. Nilai kosong tidak langsung dianggap sebagai kesalahan atau dihapus.

In [5]:
issues = []

def add_issue(rec, code_name, *, function_name="", argument_name="", detail=""):
    issues.append({
        "row_key": rec["row_key"], "language_code": rec["language_code"],
        "record_id": rec["record_id"], "issue_type": code_name,
        "function_name": function_name, "argument_name": argument_name,
        "detail": detail,
    })

# Membaca ulang hanya untuk audit schema, tanpa menyimpan katalog yang besar di RAM.
rec_by_key = {r["row_key"]: r for r in records}
for lang in LANGUAGES:
    with (RAW/lang/"input.jsonl").open("r", encoding="utf-8-sig") as fi:
        for line_no, line in enumerate(fi, 1):
            rec = rec_by_key[f"{lang}:{line_no:07d}"]
            full = json.loads(line)
            functions = full["function"]
            names = [fn.get("name") for fn in functions if isinstance(fn, dict)]
            if len(names) != len(functions) or not all(isinstance(x, str) and x for x in names):
                add_issue(rec, "invalid_function_catalog_item")
            repeats = [n for n,c in Counter(names).items() if c>1]
            if repeats:
                add_issue(rec, "duplicate_tool_names", detail=",".join(map(str, repeats)))
            by_name = {fn.get("name"): fn for fn in functions if isinstance(fn, dict) and isinstance(fn.get("name"), str)}
            if len(rec["gt"]) != 1:
                add_issue(rec, "target_function_count_not_one", detail=str(len(rec["gt"])))
            for target_name, params in rec["gt"].items():
                fn = by_name.get(target_name)
                if fn is None:
                    add_issue(rec, "target_tool_not_offered", function_name=str(target_name))
                    continue
                schema = fn.get("parameters")
                if not isinstance(schema, dict):
                    add_issue(rec, "invalid_tool_parameters_schema", function_name=target_name)
                    continue
                props = schema.get("properties")
                required = schema.get("required", [])
                if not isinstance(props, dict) or not isinstance(required, list):
                    add_issue(rec, "invalid_tool_properties_or_required", function_name=target_name)
                    continue
                if not isinstance(params, dict):
                    add_issue(rec, "gt_parameters_not_object", function_name=target_name)
                    continue
                for k in params:
                    if k not in props:
                        add_issue(rec, "gt_argument_out_of_schema", function_name=target_name, argument_name=k,
                                  detail="Kunci GT tidak tercantum dalam properties katalog tool")
                    if not isinstance(params[k], list):
                        add_issue(rec, "gt_argument_not_list", function_name=target_name, argument_name=k)
                    elif any(not isinstance(value, str) for value in params[k]):
                        add_issue(rec, "gt_argument_nonstring_value", function_name=target_name, argument_name=k)
                for required_key in required:
                    if required_key not in params:
                        add_issue(rec, "required_argument_missing", function_name=target_name, argument_name=str(required_key))

schema_issues = pd.DataFrame(issues, columns=["row_key","language_code","record_id","issue_type","function_name","argument_name","detail"])
print("Jumlah temuan skema:", len(schema_issues))
if len(schema_issues):
    display(schema_issues.groupby(["language_code","issue_type"]).size().rename("count").reset_index())
    display(schema_issues.head(15))
else:
    print("Tidak ditemukan kandidat masalah skema otomatis.")

Jumlah temuan skema: 2


,language_code,issue_type,count
0,en-US,gt_argument_out_of_schema,1
1,id-ID,gt_argument_out_of_schema,1


,row_key,language_code,record_id,issue_type,function_name,argument_name,detail
0,en-US:0001212,en-US,10061,gt_argument_out_of_schema,cooking.recipe,meal_type,Kunci GT tidak tercantum dalam properties kata...
1,id-ID:0000826,id-ID,10061,gt_argument_out_of_schema,cooking.recipe,meal_type,Kunci GT tidak tercantum dalam properties kata...


## **5 · Pemeriksaan Duplikasi dan Konflik Ground Truth**

---

Pertanyaan yang sama diperiksa berdasarkan fungsi dan argumen dalam `gt`. Perbedaan ground truth belum tentu menunjukkan kesalahan, sehingga kasus konflik ditandai untuk review dan tidak langsung dihapus.

In [6]:
# Duplikat ID dalam satu bahasa merupakan tanda bahaya, berbeda dari ID sama antarbahasa.
id_dup_groups = meta.groupby(["language_code", "record_id"], sort=True).filter(lambda df: len(df) > 1)
if len(id_dup_groups):
    for rec in records:
        if (rec["language_code"], rec["record_id"]) in set(zip(id_dup_groups.language_code, id_dup_groups.record_id)):
            add_issue(rec, "duplicate_id_within_language")
    schema_issues = pd.DataFrame(issues, columns=schema_issues.columns)

conflict_keys = set()
conflict_rows = []
for (lang, q_cmp), sub in meta.groupby(["language_code", "question_cmp"]):
    if not q_cmp or len(sub) < 2:
        continue
    if sub.gt_signature.nunique() > 1:
        key_list = sorted(sub.row_key.tolist())
        conflict_keys.update(key_list)
        for rec in records:
            if rec["row_key"] in key_list:
                conflict_rows.append({
                    "language_code": lang, "row_key": rec["row_key"],
                    "record_id": rec["record_id"], "question": rec["question"],
                    "gt_signature": rec["gt_signature"],
                    "matched_row_keys": ";".join(key_list),
                    "issue_type": "same_question_different_gt",
                })

conflicts = pd.DataFrame(conflict_rows, columns=["language_code","row_key","record_id","question","gt_signature","matched_row_keys","issue_type"])
print("Grup pertanyaan identik namun GT berbeda:", conflicts.matched_row_keys.nunique() if len(conflicts) else 0)
print("Jumlah baris yang membutuhkan review duplikat:", len(conflicts))
if len(conflicts):
    display(conflicts[["language_code","record_id","question","gt_signature","matched_row_keys"]].head(12))

print("Catatan: pertanyaan yang sama dan GT sama tetap ada, tidak didedup otomatis.")

Grup pertanyaan identik namun GT berbeda: 2
Jumlah baris yang membutuhkan review duplikat: 4


,language_code,record_id,question,gt_signature,matched_row_keys
0,id-ID,2690,matikan stopkontak,"{""iot.wemo_off"":{""device_type"":[""stopkontak""]}}",id-ID:0000233;id-ID:0000441
1,id-ID,4992,matikan stopkontak,"{""iot.wemo_off"":{""device_type"":[""""]}}",id-ID:0000233;id-ID:0000441
2,id-ID,448,nyalakan mesin pembuat kopi,"{""iot.coffee"":{""coffee_type"":["""",""cappuccino""]...",id-ID:0000055;id-ID:0000456
3,id-ID,5149,nyalakan mesin pembuat kopi,"{""iot.coffee"":{""coffee_type"":["""",""cappuccino""]...",id-ID:0000055;id-ID:0000456


Catatan: pertanyaan yang sama dan GT sama tetap ada, tidak didedup otomatis.


## **6 · Pengelompokan Data untuk Mencegah Data Leakage**

---

Data dengan ID yang sama antarbahasa atau pertanyaan identik dalam bahasa yang sama dimasukkan ke satu grup agar tidak tersebar antara *development* dan *test*.

Pengelompokan ini mengurangi risiko *data leakage*, tetapi belum mencakup semua parafrasa dengan makna serupa. Variasi data tambahan nantinya harus mengikuti grup dari data asal.

In [7]:
parents = {r["row_key"]: r["row_key"] for r in records}

def root_of(k):
    while parents[k] != k:
        parents[k] = parents[parents[k]]
        k = parents[k]
    return k

def union(a, b):
    ra, rb = root_of(a), root_of(b)
    if ra != rb:
        parents[max(ra, rb)] = min(ra, rb)

by_id = defaultdict(list)
by_question = defaultdict(list)
for r in records:
    by_id[r["record_id"]].append(r["row_key"])
    by_question[(r["language_code"], r["question_cmp"])].append(r["row_key"])
for groups in (by_id, by_question):
    for keys in groups.values():
        for other in keys[1:]:
            union(keys[0], other)

components = defaultdict(list)
for r in records:
    components[root_of(r["row_key"])].append(r["row_key"])

key_to_group = {}
group_rows = []
for members in components.values():
    members = sorted(members)
    group_id = "grp_" + hashlib.sha256("|".join(members).encode("utf-8")).hexdigest()[:16]
    for key in members:
        key_to_group[key] = group_id
    targets = [rec_by_key[key]["gt_names"] for key in members]
    target_str = [";".join(t) for t in targets]
    dominant_target = Counter(target_str).most_common(1)[0][0]
    group_rows.append({"group_id": group_id, "group_size": len(members),
                       "dominant_target": dominant_target,
                       "languages": ";".join(sorted({rec_by_key[k]["language_code"] for k in members})),
                       "row_keys": ";".join(members)})

group_df = pd.DataFrame(group_rows).sort_values("group_id").reset_index(drop=True)
meta["group_id"] = meta.row_key.map(key_to_group)
assert meta.group_id.notna().all()
print("Jumlah grup independen berdasarkan aturan yang diketahui:", len(group_df))
print("Grup >1 record:", int((group_df.group_size > 1).sum()))
print("Ukuran grup terbesar:", int(group_df.group_size.max()))
display(group_df.group_size.value_counts().sort_index().rename_axis("group_size").rename("group_count").reset_index())

Jumlah grup independen berdasarkan aturan yang diketahui: 1925
Grup >1 record: 1307
Ukuran grup terbesar: 8


,group_size,group_count
0,1,618
1,2,1286
2,3,1
3,4,15
4,6,3
5,8,2


## **7 · Pembagian Dataset Development dan Test**

---

Pembagian dilakukan berdasarkan grup, dengan target 30% untuk *development* dan 70% untuk *test*. Distribusi fungsi ikut dipertimbangkan, tetapi jumlah data setiap fungsi belum tentu seimbang.

Data yang memerlukan review tetap dicatat dalam rencana pembagian, tetapi dipisahkan dari data yang sementara layak digunakan (`eligible_provisional`).

In [8]:
rng = random.Random(RANDOM_SEED)
group_split = {}
for label, subset in group_df.groupby("dominant_target", sort=True):
    ids = sorted(subset.group_id.tolist())
    rng.shuffle(ids)
    n = len(ids)
    n_dev = 0 if n == 1 else max(1, min(n-1, round(n * DEV_GROUP_FRACTION)))
    for group_id in ids[:n_dev]: group_split[group_id] = "development"
    for group_id in ids[n_dev:]: group_split[group_id] = "test"

meta["split"] = meta.group_id.map(group_split)
group_df["split"] = group_df.group_id.map(group_split)
assert set(meta.split) == {"development", "test"}
assert meta.groupby("group_id").split.nunique().max() == 1

# Catat temuan kualitas; kebijakan: tahan satu grup penuh bila ada masalah.
unsafe_row_keys = set(schema_issues.row_key if len(schema_issues) else []) | conflict_keys
unsafe_groups = {key_to_group[k] for k in unsafe_row_keys}
meta["review_status"] = meta.group_id.map(lambda g: "review_required" if g in unsafe_groups else "eligible_provisional")
meta["source_variant"] = "original"
group_df["review_status"] = group_df.group_id.map(lambda g: "review_required" if g in unsafe_groups else "eligible_provisional")

print("Status baris:")
display(meta.groupby(["language_code", "split", "review_status"]).size().rename("records").reset_index())
print("Jumlah group yang ditahan untuk review:", len(unsafe_groups))
print("CATATAN: eligible_provisional berarti lulus audit otomatis, BUKAN validasi semantik final.")

Status baris:


,language_code,split,review_status,records
0,en-US,development,eligible_provisional,582
1,en-US,development,review_required,1
2,en-US,test,eligible_provisional,1365
3,en-US,test,review_required,4
4,id-ID,development,eligible_provisional,397
5,id-ID,development,review_required,1
6,id-ID,test,eligible_provisional,933
7,id-ID,test,review_required,4


Jumlah group yang ditahan untuk review: 3
CATATAN: eligible_provisional berarti lulus audit otomatis, BUKAN validasi semantik final.


## **8 · Pemeriksaan Hasil Pembagian Dataset**

---

ID yang sama, pertanyaan identik, dan anggota satu grup tidak boleh masuk ke split berbeda. Distribusi fungsi juga diperiksa karena fungsi yang jarang muncul mungkin hanya tersedia pada salah satu split.

In [9]:
assert meta.groupby("record_id").split.nunique().max() == 1, "Ada ID yang bocor lintas split"
assert meta.groupby(["language_code", "question_cmp"]).split.nunique().max() == 1, "Ada pertanyaan identik lintas split"
assert meta.groupby("group_id").split.nunique().max() == 1, "Grup bocor lintas split"
assert meta.row_key.is_unique and not meta.split.isna().any()

split_summary = (meta.groupby(["language_code","split","review_status"])
                 .size().rename("records").reset_index())
display(split_summary)
function_coverage = (meta.assign(primary_target=meta.target_function)
                     .groupby(["language_code", "split"])["primary_target"]
                     .nunique().rename("unique_target_functions").reset_index())
display(function_coverage)

# Daftar fungsi dengan nol contoh pada salah satu split (menghindari klaim generalisasi berlebihan).
coverage = pd.crosstab([meta.language_code, meta.target_function], meta.split)
for col in ("development", "test"):
    if col not in coverage: coverage[col] = 0
coverage = coverage.reset_index()
single_split_functions = coverage.loc[(coverage["development"]==0) | (coverage["test"]==0)]
print("Fungsi tanpa contoh pada salah satu split (per bahasa):", len(single_split_functions))
if len(single_split_functions): display(single_split_functions.head(20))

# Bagian issue review bukan data eksperimen final.
assert meta.loc[meta.review_status.eq("review_required")].group_id.isin(unsafe_groups).all()
print("Pemeriksaan anti-leakage berbasis aturan: LULUS")

,language_code,split,review_status,records
0,en-US,development,eligible_provisional,582
1,en-US,development,review_required,1
2,en-US,test,eligible_provisional,1365
3,en-US,test,review_required,4
4,id-ID,development,eligible_provisional,397
5,id-ID,development,review_required,1
6,id-ID,test,eligible_provisional,933
7,id-ID,test,review_required,4


,language_code,split,unique_target_functions
0,en-US,development,55
1,en-US,test,55
2,id-ID,development,54
3,id-ID,test,55


Fungsi tanpa contoh pada salah satu split (per bahasa): 1


split,language_code,target_function,development,test
84,id-ID,music.dislikeness,0,3


Pemeriksaan anti-leakage berbasis aturan: LULUS


## **9 · Persiapan Data untuk Review Manual**

---

Data dengan masalah skema atau ground truth berbeda dimasukkan ke daftar review bersama anggota grup terkait. Keputusan pemeriksaan dicatat dalam `manual_resolution_template.csv`.

File anotasi yang sudah tersedia tidak ditimpa saat notebook dijalankan ulang. Perbaikan tidak dilakukan langsung pada `data/raw`.

In [10]:
row_issue_index = defaultdict(list)
for item in issues:
    row_issue_index[item["row_key"]].append(item["issue_type"] + (":" + item["argument_name"] if item["argument_name"] else ""))
for item in conflict_rows:
    row_issue_index[item["row_key"]].append("same_question_different_gt")

review_queue_rows = []
for r in records:
    if key_to_group[r["row_key"]] not in unsafe_groups:
        continue
    review_queue_rows.append({
        "row_key": r["row_key"], "language_code": r["language_code"],
        "record_id": r["record_id"], "group_id": key_to_group[r["row_key"]],
        "planned_split": meta.set_index("row_key").at[r["row_key"], "split"],
        "question": r["question"], "gt_json": r["gt_signature"],
        "issue_on_this_row": "; ".join(sorted(set(row_issue_index[r["row_key"]]))),
        "issue_in_group": "yes",
    })
review_queue = pd.DataFrame(review_queue_rows, columns=[
    "row_key","language_code","record_id","group_id","planned_split","question","gt_json","issue_on_this_row","issue_in_group"
])
display(review_queue.head(20))

manual_path = OUT / "manual_resolution_template.csv"
if not manual_path.exists():
    manual_template = review_queue.copy()
    for field in ("review_decision", "corrected_gt_json", "reviewer", "review_date", "decision_reason"):
        manual_template[field] = ""
    manual_template.to_csv(manual_path, index=False, encoding="utf-8-sig")
    print("Template manual dibuat:", manual_path.name)
else:
    print("Template manual sudah ada dan TIDAK ditimpa:", manual_path.name)
print("Baris review wajib (termasuk anggota kelompok yang ikut ditahan):", len(review_queue))

,row_key,language_code,record_id,group_id,planned_split,question,gt_json,issue_on_this_row,issue_in_group
0,en-US:0000070,en-US,448,grp_da1a8ef6e3b59760,test,run coffee maker,"{""iot.coffee"":{""coffee_type"":["""",""cappuccino""]...",,yes
1,en-US:0000343,en-US,2690,grp_a05dc02aca931212,test,turn off the socket,"{""iot.wemo_off"":{""device_type"":[""socket""]}}",,yes
2,en-US:0000648,en-US,4992,grp_a05dc02aca931212,test,turn off sockets,"{""iot.wemo_off"":{""device_type"":[""""]}}",,yes
3,en-US:0000668,en-US,5149,grp_da1a8ef6e3b59760,test,set the coffee maker to on,"{""iot.coffee"":{""coffee_type"":["""",""cappuccino""]...",,yes
4,en-US:0001212,en-US,10061,grp_7c8b1eb0fdab22ab,development,what should i make for dinner,"{""cooking.recipe"":{""app_name"":[""""],""cooking_ty...",gt_argument_out_of_schema:meal_type,yes
5,id-ID:0000055,id-ID,448,grp_da1a8ef6e3b59760,test,nyalakan mesin pembuat kopi,"{""iot.coffee"":{""coffee_type"":["""",""cappuccino""]...",same_question_different_gt,yes
6,id-ID:0000233,id-ID,2690,grp_a05dc02aca931212,test,matikan stopkontak,"{""iot.wemo_off"":{""device_type"":[""stopkontak""]}}",same_question_different_gt,yes
7,id-ID:0000441,id-ID,4992,grp_a05dc02aca931212,test,matikan stopkontak,"{""iot.wemo_off"":{""device_type"":[""""]}}",same_question_different_gt,yes
8,id-ID:0000456,id-ID,5149,grp_da1a8ef6e3b59760,test,nyalakan mesin pembuat kopi,"{""iot.coffee"":{""coffee_type"":["""",""cappuccino""]...",same_question_different_gt,yes
9,id-ID:0000826,id-ID,10061,grp_7c8b1eb0fdab22ab,development,apa yang perlu saya buat untuk makan malam,"{""cooking.recipe"":{""app_name"":[""""],""cooking_ty...",gt_argument_out_of_schema:meal_type,yes


Template manual sudah ada dan TIDAK ditimpa: manual_resolution_template.csv
Baris review wajib (termasuk anggota kelompok yang ikut ditahan): 10


## **10 · Penyimpanan Dataset Hasil Preprocessing**

---

Data sementara yang lolos pemeriksaan otomatis disimpan terpisah untuk *development* dan *test*. Data yang masih perlu diperiksa disimpan di `review_required.jsonl`.

Teks perintah, `gt`, dan katalog fungsi asli tetap dipertahankan tanpa normalisasi atau pengurangan jumlah fungsi.

In [11]:
audit_index = meta.set_index("row_key", drop=False)
output_paths = {}
for lang in LANGUAGES:
    for split in ("development", "test"):
        output_paths[(lang, split)] = PROCESSED / f"{lang}_{split}.jsonl"
output_paths["review"] = PROCESSED / "review_required.jsonl"

counts_written = Counter()
# Tulis ulang dengan urutan sumber agar dapat ditelusuri lewat line_number.
from contextlib import ExitStack
with ExitStack() as stack:
    handles = {key: stack.enter_context(path.open("w", encoding="utf-8", newline="\n")) for key,path in output_paths.items()}
    for lang in LANGUAGES:
        with (RAW/lang/"input.jsonl").open("r", encoding="utf-8-sig") as fh:
            for line_no, line in enumerate(fh, 1):
                source = json.loads(line)
                row_key = f"{lang}:{line_no:07d}"
                info = audit_index.loc[row_key]
                new_row = {
                    "row_key": row_key,
                    "source_record_id": str(source["id"]),
                    "language_code": lang,
                    "source_language_field": source.get("language"),
                    "source_variant": "original",
                    "group_id": info["group_id"],
                    "split": info["split"],
                    "review_status": info["review_status"],
                    "question": source["question"],
                    "function": source["function"],
                    "gt": source["gt"],
                    "test_category": source.get("test_category"),
                }
                key = "review" if info["review_status"] == "review_required" else (lang, info["split"])
                handles[key].write(json.dumps(new_row, ensure_ascii=False) + "\n")
                counts_written[str(key)] += 1

print("Ekspor selesai:")
for key, p in output_paths.items():
    print(f"- {p.name}: {counts_written[str(key)]} record; {p.stat().st_size/(1024**2):.1f} MiB")
print("Tidak ada modifikasi pada file dataset asli.")

Ekspor selesai:
- en-US_development.jsonl: 582 record; 23.4 MiB
- en-US_test.jsonl: 1365 record; 54.9 MiB
- id-ID_development.jsonl: 397 record; 16.0 MiB
- id-ID_test.jsonl: 933 record; 37.5 MiB
- review_required.jsonl: 10 record; 0.4 MiB
Tidak ada modifikasi pada file dataset asli.


## **11 · Validasi Hasil Ekspor**

---

Jumlah data hasil ekspor diperiksa agar tidak ada baris hilang atau ganda. Kecocokan pertanyaan, ground truth, grup, dan checksum file sumber juga diperiksa sebelum data digunakan.

In [12]:
seen = set()
count_exported = 0
for key, path in output_paths.items():
    with path.open("r", encoding="utf-8") as fh:
        for line in fh:
            row = json.loads(line)
            row_key = row["row_key"]
            assert row_key not in seen, f"Row key ganda: {row_key}"
            seen.add(row_key)
            count_exported += 1
            assert row["question"] == rec_by_key[row_key]["question"]
            assert row["gt"] == rec_by_key[row_key]["gt"]
            assert row["group_id"] == key_to_group[row_key]
            if key == "review":
                assert row["review_status"] == "review_required"
            else:
                assert row["review_status"] == "eligible_provisional"
                assert (row["language_code"], row["split"]) == key

assert count_exported == len(records) == len(seen), "Ada baris hilang/bertambah pada ekspor"
for r in raw_manifest.itertuples(index=False):
    assert sha256_file(RAW/r.language_code/r.filename) == r.sha256, "File raw berubah selama proses"

print("Validasi PASCA-EKSPOR: LULUS")
print("Record sumber:", len(records), "| total record output:", count_exported,
      "| eligible:", int(meta.review_status.eq("eligible_provisional").sum()),
      "| review:", int(meta.review_status.eq("review_required").sum()))

Validasi PASCA-EKSPOR: LULUS
Record sumber: 3287 | total record output: 3287 | eligible: 3277 | review: 10


## **12 · Kesimpulan Hasil Preprocessing**

---

### **Ringkasan Hasil Analisis**

Hasil perhitungan ditampilkan dalam tabel. Data dengan status `eligible_provisional` masih memerlukan keputusan evaluasi dan bukan berarti seluruh anotasinya sudah tervalidasi.

In [13]:
raw_manifest.to_csv(OUT/"raw_manifest.csv", index=False, encoding="utf-8-sig")
schema_issues.to_csv(OUT/"schema_issues.csv", index=False, encoding="utf-8-sig")
conflicts.to_csv(OUT/"duplicate_gt_conflicts.csv", index=False, encoding="utf-8-sig")
group_df.to_csv(OUT/"leakage_groups.csv", index=False, encoding="utf-8-sig")
review_queue.to_csv(OUT/"review_queue.csv", index=False, encoding="utf-8-sig")
split_summary.to_csv(OUT/"split_summary.csv", index=False, encoding="utf-8-sig")
coverage.to_csv(OUT/"function_split_coverage.csv", index=False, encoding="utf-8-sig")
meta[["row_key","record_id","language_code","line_number","question","target_function",
      "group_id","split","review_status","source_variant"]].to_csv(
          OUT/"split_manifest.csv", index=False, encoding="utf-8-sig")

n_review = int(meta.review_status.eq("review_required").sum())
n_ready = int(meta.review_status.eq("eligible_provisional").sum())
summary = {
    "project": "KP Agentic AI + LLM",
    "dataset": DATASET_FOLDER,
    "language_codes": LANGUAGES,
    "seed": RANDOM_SEED,
    "dev_group_fraction": DEV_GROUP_FRACTION,
    "split_method": "connected components of cross-locale record_id and within-locale exact normalized question, then stratified seeded group assignment by dominant target",
    "counts_by_language": {l: int(meta.language_code.eq(l).sum()) for l in LANGUAGES},
    "total_source_records": len(records),
    "total_groups": len(group_df),
    "schema_issue_entries": len(schema_issues),
    "duplicate_conflict_groups": int(conflicts.matched_row_keys.nunique()) if len(conflicts) else 0,
    "review_required_records": n_review,
    "eligible_provisional_records": n_ready,
    "candidate_function_policy": "preserve per-record original function list; full 55 tools, no automatic 8-tool selection",
    "manual_language_review_completed": False,
    "manual_translation_review_completed": False,
    "baseline_ready_final": False,
    "raw_file_sha256": {f"{r.language_code}/{r.filename}": r.sha256 for r in raw_manifest.itertuples(index=False)},
    "output_files": [str(p.relative_to(ROOT)) for p in output_paths.values()],
}
(OUT/"preprocessing_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

# Tabel hasil perhitungan — interpretasi ditulis terpisah di Markdown.
def jumlah_status(lang, split, status="eligible_provisional"):
    return int((
        meta.language_code.eq(lang)
        & meta.split.eq(split)
        & meta.review_status.eq(status)
    ).sum())

n_conflict_groups = int(conflicts.matched_row_keys.nunique()) if len(conflicts) else 0
hasil_preprocessing = pd.DataFrame([
    {
        "Indikator": "Jumlah data asli",
        "Hasil": ", ".join(
            f"{lang}: {int(meta.language_code.eq(lang).sum()):,}"
            for lang in LANGUAGES
        ) + f" (total {len(records):,})",
    },
    {
        "Indikator": "Pemeriksaan struktur JSONL",
        "Hasil": "Lulus, tidak ditemukan masalah pembacaan atau pasangan input–golden",
    },
    {
        "Indikator": "Warning skema GT",
        "Hasil": f"{len(schema_issues)} temuan yang perlu diperiksa",
    },
    {
        "Indikator": "Duplikasi dengan GT berbeda",
        "Hasil": f"{n_conflict_groups} grup",
    },
    {
        "Indikator": "Grup anti-leakage",
        "Hasil": f"{len(group_df):,} grup",
    },
    {
        "Indikator": "Development (eligible)",
        "Hasil": ", ".join(
            f"{lang}: {jumlah_status(lang, 'development'):,}"
            for lang in LANGUAGES
        ),
    },
    {
        "Indikator": "Test (eligible)",
        "Hasil": ", ".join(
            f"{lang}: {jumlah_status(lang, 'test'):,}"
            for lang in LANGUAGES
        ),
    },
    {
        "Indikator": "Eligible provisional",
        "Hasil": f"{n_ready:,} data",
    },
    {
        "Indikator": "Perlu review manual",
        "Hasil": f"{n_review:,} data",
    },
    {
        "Indikator": "Validasi pasca-ekspor",
        "Hasil": f"Lulus, {len(records):,} data tercatat dan checksum sumber tetap sama",
    },
])

display(hasil_preprocessing)

# Laporan angka disimpan otomatis; insight ditulis pada Markdown notebook.
report = "# Ringkasan Hasil Preprocessing — MASSIVE-Agents\n\n"
report += "| Indikator | Hasil |\n|---|---|\n"
for row in hasil_preprocessing.itertuples(index=False):
    report += f"| {row.Indikator} | {row.Hasil} |\n"
(OUT / "preprocessing_report.md").write_text(report + "\n", encoding="utf-8")

print("\nRingkasan disimpan di:", OUT / "preprocessing_report.md")
print("Dataset hasil preprocessing:", PROCESSED)


,Indikator,Hasil
0,Jumlah data asli,"en-US: 1,952, id-ID: 1,335 (total 3,287)"
1,Pemeriksaan struktur JSONL,"Lulus, tidak ditemukan masalah pembacaan atau ..."
2,Warning skema GT,2 temuan yang perlu diperiksa
3,Duplikasi dengan GT berbeda,2 grup
4,Grup anti-leakage,"1,925 grup"
5,Development (eligible),"en-US: 582, id-ID: 397"
6,Test (eligible),"en-US: 1,365, id-ID: 933"
7,Eligible provisional,"3,277 data"
8,Perlu review manual,10 data
9,Validasi pasca-ekspor,"Lulus, 3,287 data tercatat dan checksum sumber..."



Ringkasan disimpan di: D:\KP-Agentic-AI\results\preprocessing\preprocessing_report.md
Dataset hasil preprocessing: D:\KP-Agentic-AI\data\processed\massive_agents



## **Insight Hasil Preprocessing**

---

| Proses | Hasil dan Insight |
|---|---|
| **Data Screening** | Sebanyak 3.277 data lolos pemeriksaan otomatis, sedangkan 10 data dipisahkan untuk review manual. |
| **Anti-Leakage Grouping** | Terbentuk 1.925 grup berdasarkan kesamaan ID dan pertanyaan untuk mencegah data terkait tersebar ke split berbeda. |
| **Group-Based Stratified Splitting** | Dataset sementara terbagi menjadi 979 data development dan 2.298 data test, dengan mempertimbangkan pengelompokan data dan distribusi fungsi target. |
| **Output Validation** | Seluruh 3.287 data tetap tercatat tanpa kehilangan data. Tidak ditemukan kebocoran antarsplit berdasarkan aturan pengelompokan yang digunakan. |

### **Langkah Selanjutnya**

1. Menyelesaikan review terhadap 10 data yang ditahan.
2. Memastikan pembagian dataset sudah sesuai kebutuhan evaluasi.
3. Menentukan konfigurasi pengujian dan metrik evaluasi model.
4. Menjalankan baseline model menggunakan dataset yang sudah disiapkan.
